In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler

print("Feature engineering notebook started.")

Feature engineering notebook started.


In [2]:
PROJECT_ROOT = Path.cwd().parents[0]

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "combined_energy_data.csv"
)

df = pd.read_csv(DATA_PATH)

print("Combined dataset loaded.")
print("Shape:", df.shape)

df.head()

Combined dataset loaded.
Shape: (2024, 9)


,id,timestamp,zone_name,appliance,people_count,duration_seconds,power_watts,energy_wh,data_source
0,1.0,2026-09-29T02:38:11.185845,Zone 1,light,2,60.000000,40.0,0.666667,real
1,2.0,2026-09-29T02:38:11.188136,Zone 1,fan,2,60.000000,75.0,1.250000,real
2,3.0,2026-09-29T02:43:57.272710,Zone 2,light,0,17.856521,40.0,0.198406,real
3,4.0,2026-09-29T02:43:57.284567,Zone 2,fan,0,15.834768,75.0,0.329891,real
4,5.0,2026-09-29T02:44:13.127614,Zone 1,light,0,16.857750,40.0,0.187308,real


In [3]:
print("Dataset information:")
print()

print(df.info())

Dataset information:

<class 'pandas.DataFrame'>
RangeIndex: 2024 entries, 0 to 2023
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                24 non-null     float64
 1   timestamp         2024 non-null   str    
 2   zone_name         2024 non-null   str    
 3   appliance         2024 non-null   str    
 4   people_count      2024 non-null   int64  
 5   duration_seconds  2024 non-null   float64
 6   power_watts       2024 non-null   float64
 7   energy_wh         2024 non-null   float64
 8   data_source       2024 non-null   str    
dtypes: float64(4), int64(1), str(4)
memory usage: 142.4 KB
None


In [4]:
print("Missing values:")
print(df.isnull().sum())

Missing values:
id                  2000
timestamp              0
zone_name              0
appliance              0
people_count           0
duration_seconds       0
power_watts            0
energy_wh              0
data_source            0
dtype: int64


In [5]:
features = df.copy()

print("Feature-engineering dataset created.")
print("Shape:", features.shape)

Feature-engineering dataset created.
Shape: (2024, 9)


In [6]:
features["timestamp"] = pd.to_datetime(
    features["timestamp"],
    format="mixed"
)

print("Timestamp converted successfully.")

Timestamp converted successfully.


In [7]:
features["hour"] = features["timestamp"].dt.hour

features["minute"] = features["timestamp"].dt.minute

features["day_of_week"] = (
    features["timestamp"].dt.dayofweek
)

features["day_of_month"] = (
    features["timestamp"].dt.day
)

features["month"] = (
    features["timestamp"].dt.month
)

features["is_weekend"] = (
    features["day_of_week"] >= 5
).astype(int)

print("Time features created.")

Time features created.


In [8]:
print(
    features[
        [
            "timestamp",
            "hour",
            "minute",
            "day_of_week",
            "day_of_month",
            "month",
            "is_weekend"
        ]
    ].head()
)

                   timestamp  hour  minute  day_of_week  day_of_month  month  \
0 2026-09-29 02:38:11.185845     2      38            1            29      9   
1 2026-09-29 02:38:11.188136     2      38            1            29      9   
2 2026-09-29 02:43:57.272710     2      43            1            29      9   
3 2026-09-29 02:43:57.284567     2      43            1            29      9   
4 2026-09-29 02:44:13.127614     2      44            1            29      9   

   is_weekend  
0           0  
1           0  
2           0  
3           0  
4           0  


In [9]:
def get_time_period(hour):

    if 5 <= hour < 12:
        return "Morning"

    elif 12 <= hour < 17:
        return "Afternoon"

    elif 17 <= hour < 21:
        return "Evening"

    else:
        return "Night"


features["time_period"] = (
    features["hour"]
    .apply(get_time_period)
)

print("Time period feature created.")

Time period feature created.


In [10]:
print(
    features["time_period"].value_counts()
)

time_period
Night        689
Morning      577
Afternoon    427
Evening      331
Name: count, dtype: int64


In [11]:
features["occupancy_level"] = np.select(
    [
        features["people_count"] == 0,
        features["people_count"].between(1, 20),
        features["people_count"] > 20
    ],
    [
        "Empty",
        "Low",
        "High"
    ],
    default="Unknown"
)

print("Occupancy level created.")

Occupancy level created.


In [12]:
features["high_occupancy"] = (
    features["people_count"] > 20
).astype(int)

print("High occupancy feature created.")

High occupancy feature created.


In [15]:
print(
    features[
        [
            "people_count",
            "occupancy_level",
            "high_occupancy"
        ]
    ].sample(20)
)

      people_count occupancy_level  high_occupancy
1895            18             Low               0
1539            17             Low               0
1220            16             Low               0
1110            17             Low               0
1411            19             Low               0
1897            26            High               1
948             18             Low               0
185             34            High               1
733              0           Empty               0
855             20             Low               0
1465            11             Low               0
721              0           Empty               0
1073             7             Low               0
778             15             Low               0
934             23            High               1
1741             9             Low               0
276             21            High               1
780              6             Low               0
1976            15             

In [16]:
features["duration_minutes"] = (
    features["duration_seconds"] / 60
)

features["duration_hours"] = (
    features["duration_seconds"] / 3600
)

features["long_session"] = (
    features["duration_seconds"] >= 30
).astype(int)

print("Duration features created.")

Duration features created.


In [17]:
print(
    features[
        [
            "duration_seconds",
            "duration_minutes",
            "duration_hours",
            "long_session"
        ]
    ].head(10)
)

   duration_seconds  duration_minutes  duration_hours  long_session
0         60.000000          1.000000        0.016667             1
1         60.000000          1.000000        0.016667             1
2         17.856521          0.297609        0.004960             0
3         15.834768          0.263913        0.004399             0
4         16.857750          0.280962        0.004683             0
5         14.828998          0.247150        0.004119             0
6         23.567099          0.392785        0.006546             0
7         21.577254          0.359621        0.005994             0
8         11.130185          0.185503        0.003092             0
9          9.131148          0.152186        0.002536             0


In [18]:
features = features.drop(
    columns=[
        "duration_minutes",
        "duration_hours"
    ]
)

print("Redundant duration features removed.")

Redundant duration features removed.


In [19]:
print("Current feature columns:")
print()

print(features.columns.tolist())

Current feature columns:

['id', 'timestamp', 'zone_name', 'appliance', 'people_count', 'duration_seconds', 'power_watts', 'energy_wh', 'data_source', 'hour', 'minute', 'day_of_week', 'day_of_month', 'month', 'is_weekend', 'time_period', 'occupancy_level', 'high_occupancy', 'long_session']


In [20]:
numeric_features = features.select_dtypes(
    include=["int64", "float64"]
)

correlation_matrix = numeric_features.corr()

print("Correlation matrix:")
correlation_matrix

Correlation matrix:


,id,people_count,duration_seconds,power_watts,energy_wh,is_weekend,high_occupancy,long_session
id,1.000000,0.244497,-0.308203,0.072232,-0.237781,NaN,NaN,-0.258423
people_count,0.244497,1.000000,0.355595,0.052940,0.315509,0.007511,0.795781,0.343461
duration_seconds,-0.308203,0.355595,1.000000,0.023739,0.842224,0.000821,0.264903,0.521522
power_watts,0.072232,0.052940,0.023739,1.000000,0.496578,-0.026237,0.054406,0.027205
energy_wh,-0.237781,0.315509,0.842224,0.496578,1.000000,-0.016350,0.243999,0.439748
is_weekend,NaN,0.007511,0.000821,-0.026237,-0.016350,1.000000,-0.001117,0.029228
high_occupancy,NaN,0.795781,0.264903,0.054406,0.243999,-0.001117,1.000000,0.217464
long_session,-0.258423,0.343461,0.521522,0.027205,0.439748,0.029228,0.217464,1.000000


In [21]:
energy_correlation = (
    correlation_matrix["energy_wh"]
    .sort_values(
        ascending=False
    )
)

print("Correlation with energy_wh:")
print(energy_correlation)

Correlation with energy_wh:
energy_wh           1.000000
duration_seconds    0.842224
power_watts         0.496578
long_session        0.439748
people_count        0.315509
high_occupancy      0.243999
is_weekend         -0.016350
id                 -0.237781
Name: energy_wh, dtype: float64


In [22]:
features_for_ml = features.drop(
    columns=[
        "id",
        "timestamp",
        "data_source",
        "energy_wh"
    ]
)

print("ML feature dataset created.")
print()

print(
    features_for_ml.columns.tolist()
)

ML feature dataset created.

['zone_name', 'appliance', 'people_count', 'duration_seconds', 'power_watts', 'hour', 'minute', 'day_of_week', 'day_of_month', 'month', 'is_weekend', 'time_period', 'occupancy_level', 'high_occupancy', 'long_session']


In [23]:
X = features_for_ml.copy()

y = features["energy_wh"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print()
print("Target:", y.name)

X shape: (2024, 15)
y shape: (2024,)

Target: energy_wh


In [24]:
categorical_columns = (
    X.select_dtypes(
        include=[
            "object",
            "string",
            "category"
        ]
    ).columns.tolist()
)

print("Categorical columns:")
print(categorical_columns)

Categorical columns:
['zone_name', 'appliance', 'time_period', 'occupancy_level']


In [25]:
X_encoded = pd.get_dummies(
    X,
    columns=categorical_columns,
    drop_first=True
)

X_encoded = X_encoded.astype(float)

print("One-hot encoding completed.")

print()
print("Encoded shape:", X_encoded.shape)

print()
print("Encoded columns:")
print(X_encoded.columns.tolist())

One-hot encoding completed.

Encoded shape: (2024, 18)

Encoded columns:
['people_count', 'duration_seconds', 'power_watts', 'hour', 'minute', 'day_of_week', 'day_of_month', 'month', 'is_weekend', 'high_occupancy', 'long_session', 'zone_name_Zone 2', 'appliance_light', 'time_period_Evening', 'time_period_Morning', 'time_period_Night', 'occupancy_level_High', 'occupancy_level_Low']


In [26]:
feature_variance = (
    X_encoded.var()
    .sort_values()
)

print("Feature variance:")
print(feature_variance)

Feature variance:
long_session               0.116023
time_period_Evening        0.136861
is_weekend                 0.201978
time_period_Morning        0.203910
high_occupancy             0.210852
occupancy_level_High       0.210852
time_period_Night          0.224644
occupancy_level_Low        0.250061
appliance_light            0.250076
zone_name_Zone 2           0.250121
month                      1.256706
day_of_week                3.978432
hour                      48.066193
day_of_month              78.550104
people_count             120.541507
minute                   301.005662
power_watts              306.342745
duration_seconds        1242.538514
dtype: float64


In [27]:
zero_variance_features = (
    feature_variance[
        feature_variance == 0
    ]
    .index
    .tolist()
)

print("Zero-variance features:")
print(zero_variance_features)

Zero-variance features:
[]


In [28]:
X_selected = X_encoded.drop(
    columns=zero_variance_features
)

print("Zero-variance features removed.")

print()
print("Final candidate feature count:")
print(X_selected.shape[1])

print()
print("Selected features:")
print(X_selected.columns.tolist())

Zero-variance features removed.

Final candidate feature count:
18

Selected features:
['people_count', 'duration_seconds', 'power_watts', 'hour', 'minute', 'day_of_week', 'day_of_month', 'month', 'is_weekend', 'high_occupancy', 'long_session', 'zone_name_Zone 2', 'appliance_light', 'time_period_Evening', 'time_period_Morning', 'time_period_Night', 'occupancy_level_High', 'occupancy_level_Low']


In [29]:
scaler = StandardScaler()

X_final = pd.DataFrame(
    scaler.fit_transform(X_selected),
    columns=X_selected.columns,
    index=X_selected.index
)

print("Feature scaling completed.")

print()
print("Final feature shape:")
print(X_final.shape)

Feature scaling completed.

Final feature shape:
(2024, 18)


In [30]:
print("Final ML features:")
X_final.head()

Final ML features:


,people_count,duration_seconds,power_watts,hour,minute,day_of_week,day_of_month,month,is_weekend,high_occupancy,long_session,zone_name_Zone 2,appliance_light,time_period_Evening,time_period_Morning,time_period_Night,occupancy_level_High,occupancy_level_Low
0,-1.086680,-0.191419,-1.013931,-1.337883,0.471676,-0.984868,1.461472,6.145295,-0.624588,-0.657581,0.393182,-0.997040,1.013931,-0.442166,-0.631471,1.391974,-0.657581,1.015937
1,-1.086680,-0.191419,0.986260,-1.337883,0.471676,-0.984868,1.461472,6.145295,-0.624588,-0.657581,0.393182,-0.997040,-0.986260,-0.442166,-0.631471,1.391974,-0.657581,1.015937
2,-1.268889,-1.387286,-1.013931,-1.337883,0.759940,-0.984868,1.461472,6.145295,-0.624588,-0.657581,-2.543351,1.002969,1.013931,-0.442166,-0.631471,1.391974,-0.657581,-0.984313
3,-1.268889,-1.444655,0.986260,-1.337883,0.759940,-0.984868,1.461472,6.145295,-0.624588,-0.657581,-2.543351,1.002969,-0.986260,-0.442166,-0.631471,1.391974,-0.657581,-0.984313
4,-1.268889,-1.415627,-1.013931,-1.337883,0.817592,-0.984868,1.461472,6.145295,-0.624588,-0.657581,-2.543351,-0.997040,1.013931,-0.442166,-0.631471,1.391974,-0.657581,-0.984313


In [31]:
print("Target values:")
y.head()

Target values:


0    0.666667
1    1.250000
2    0.198406
3    0.329891
4    0.187308
Name: energy_wh, dtype: float64

In [32]:
PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

X_final.to_csv(
    PROCESSED_DIR
    / "model_features.csv",
    index=False
)

y.to_csv(
    PROCESSED_DIR
    / "model_target.csv",
    index=False
)

print("Model-ready data saved.")

print()
print(
    "Features:",
    PROCESSED_DIR / "model_features.csv"
)

print(
    "Target:",
    PROCESSED_DIR / "model_target.csv"
)

Model-ready data saved.

Features: c:\Users\Devanshu Lingwal\OneDrive\Desktop\Personal\smart-classroom-automation\data\processed\model_features.csv
Target: c:\Users\Devanshu Lingwal\OneDrive\Desktop\Personal\smart-classroom-automation\data\processed\model_target.csv


In [33]:
print("========================================")
print("FEATURE ENGINEERING COMPLETE")
print("========================================")

print()

print("Original dataset:")
print(df.shape)

print()

print("Final ML features:")
print(X_final.shape)

print()

print("Target:")
print(y.shape)

print()

print("Feature columns:")

for column in X_final.columns:
    print("-", column)

FEATURE ENGINEERING COMPLETE

Original dataset:
(2024, 9)

Final ML features:
(2024, 18)

Target:
(2024,)

Feature columns:
- people_count
- duration_seconds
- power_watts
- hour
- minute
- day_of_week
- day_of_month
- month
- is_weekend
- high_occupancy
- long_session
- zone_name_Zone 2
- appliance_light
- time_period_Evening
- time_period_Morning
- time_period_Night
- occupancy_level_High
- occupancy_level_Low
